
# 14 — LangGraph: AI-Driven Decision Workflows

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from course topics:**
- LangGraph state, nodes and edges
- Conditional routing
- Chat models and prompt templates
- Stateful graph execution and checkpointing
- Workflow inspection and verification

**LangGraph documentation:**
https://docs.langchain.com/oss/python/langgraph/overview

## Goal

Build a customer-support workflow where a real
language model decides which processing path to take.

The workflow:

1. Receives a customer message.
2. Uses a structured classification prompt.
3. Calls Gemini to select a support category.
4. Validates the returned category.
5. Routes execution through LangGraph.
6. Selects the corresponding local policy.
7. Produces a policy-based response.
8. Saves the execution state and verifies the result.

The model performs classification.

LangGraph controls the execution flow.

Python validates the model decision and constructs
the final policy response.

This example uses two fictional customer requests
and two Gemini model calls.

It does not execute customer transactions or modify
external systems.



## 1. Why LangGraph for Prompt Engineering?

A prompt can instruct a language model to classify
a request, extract information or select an action.

But a prompt alone does not manage the entire
application workflow.

LangGraph connects model decisions to processing steps.

Important concepts:

**State:** Information shared between workflow nodes.

**Node:** A function that performs one step.

**Edge:** A connection between steps.

**Conditional edge:** Selects the next step according
to information in the state.

**Checkpointer:** Saves snapshots of execution state.

**Thread ID:** Identifies one workflow execution history.

In our example, Gemini's prompt determines a category,
and LangGraph uses that category to choose a branch.

Unlike simple keyword classification, the model can
interpret different ways of describing the same issue.

We will test that ability on two synthetic examples.


In [1]:


%pip install -q "langgraph>=1,<2" "langchain-core>=1,<2" "langchain-openai>=1,<2"



## 2. Define the Task and Shared State

Our fictional support team handles four categories:

- `returns`: Returns and refunds.
- `delivery`: Parcel tracking and delivery issues.
- `damage`: Broken or damaged products.
- `general`: Requests that do not fit the other categories.

Each category has a predefined policy.

The workflow state will hold:

- The original message.
- Gemini's selected category.
- The selected policy.
- The final response.
- An execution log.

We use `TypedDict` to describe these fields.

The final response is constructed from a predefined
policy, not generated by another Gemini request.

This separates model decision-making from the
application's business rules.


In [2]:

from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


class SupportState(TypedDict, total=False):
    message: str
    category: str
    policy: str
    answer: str
    log: Annotated[list[str], add]


POLICIES = {
    "returns": (
        "Unused products may be returned within "
        "30 days with a receipt."
    ),
    "delivery": (
        "Standard delivery usually takes "
        "3 to 5 business days. "
        "Contact support for help with delayed parcels."
    ),
    "damage": (
        "Customers who receive damaged items "
        "should contact support to discuss "
        "replacement options."
    ),
    "general": (
        "Contact customer support for assistance "
        "with questions not covered by these policies."
    )
}

ALLOWED_CATEGORIES = set(POLICIES)

print("Graph state defined.")
print("Categories:", sorted(ALLOWED_CATEGORIES))
print("Model requests: 0")


Graph state defined.
Categories: ['damage', 'delivery', 'general', 'returns']
Model requests: 0



## 3. Design the Classification Prompt

This is the primary prompt-engineering component.

Gemini must choose one of four predefined categories.

A good classification prompt should clearly specify:

- The model's role.
- The allowed categories.
- The meaning of each category.
- The expected output format.
- What to do when no category applies.

We require the model to return exactly one
lowercase category name.

For example:

Input:
"My package hasn't shown up."

Expected category:
`delivery`

This input is useful because it describes a delivery
problem without using the word "delivery".

The model's output is validated before LangGraph
uses it for routing.

An invalid category is rejected rather than silently
used to execute an unintended branch.


In [3]:

from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

gemini = ChatOpenAI(
    model=MODEL,
    api_key=gemini_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    temperature=0,
    max_retries=0,
    timeout=120,
    use_responses_api=False
)


classification_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a customer-support request classifier.

Choose exactly ONE category:

returns:
Requests to return a product or obtain a refund.

delivery:
Questions about shipping, parcels, tracking,
non-arrival or delayed delivery.

damage:
Products that arrived broken, defective or damaged.

general:
Requests that do not belong to the categories above.

Rules:
- Return only one category name.
- Use lowercase letters.
- Do not add explanations or punctuation.
- Do not invent new categories.
- Select the category describing the main issue.
"""
    ),
    (
        "human",
        "Customer message:\n{message}"
    )
])


classification_chain = (
    classification_prompt
    | gemini
    | StrOutputParser()
)

print("Classification prompt ready.")
print("Gemini model:", MODEL)
print("Model requests: 0")


Enter your Gemini API key: ··········
Classification prompt ready.
Gemini model: gemini-3.5-flash-lite
Model requests: 0



## 4. Create the LangGraph Nodes

We need three kinds of operations.

**Classification node**

Calls Gemini and validates the category.

**Policy nodes**

Each branch selects its corresponding local policy.

**Finalization node**

Creates a response using the selected policy.

The conditional edge uses Gemini's validated
category to choose one of four policy branches.

This makes the model's decision operational:
different classifications produce different
execution paths.

The policy nodes are deliberately deterministic.
A language model is not needed to look up a
predefined policy string.


In [4]:

def classify_with_gemini(state: SupportState):
    raw_category = classification_chain.invoke({
        "message": state["message"]
    })

    category = raw_category.strip().lower()

    if category not in ALLOWED_CATEGORIES:
        raise ValueError(
            f"Gemini returned an invalid category: "
            f"{raw_category!r}"
        )

    return {
        "category": category,
        "log": [f"Gemini classified request: {category}"]
    }


def make_policy_node(category):
    def select_policy(state: SupportState):
        return {
            "policy": POLICIES[category],
            "log": [f"Executed {category} policy node"]
        }

    return select_policy


def route_by_category(state: SupportState):
    return state["category"]


def finalize_response(state: SupportState):
    return {
        "answer": (
            "According to the relevant support policy: "
            + state["policy"]
        ),
        "log": ["Final response constructed"]
    }


print("Classification and policy nodes defined.")
print("Model requests: 0")


Classification and policy nodes defined.
Model requests: 0


In [5]:

builder = StateGraph(SupportState)

builder.add_node("classify", classify_with_gemini)

for category in POLICIES:
    builder.add_node(
        category,
        make_policy_node(category)
    )

builder.add_node("finalize", finalize_response)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "returns": "returns",
        "delivery": "delivery",
        "damage": "damage",
        "general": "general"
    }
)

for category in POLICIES:
    builder.add_edge(category, "finalize")

builder.add_edge("finalize", END)

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)

print("AI-driven LangGraph compiled.")
print("Graph nodes:", list(graph.get_graph().nodes))
print("Model requests: 0")


AI-driven LangGraph compiled.
Graph nodes: ['__start__', 'classify', 'returns', 'delivery', 'damage', 'general', 'finalize', '__end__']
Model requests: 0



## 5. Execute Two AI-Driven Workflows

We will test two different customer messages.

**Case 1 — Delivery**

"My package hasn't shown up."

Expected category: `delivery`

**Case 2 — Damage**

"My headphones arrived in pieces."

Expected category: `damage`

The examples deliberately avoid the exact category
names to test whether Gemini can recognize the
underlying meaning.

Each case uses a separate LangGraph thread.

Each successful graph execution makes one Gemini
classification request.

The policy lookup and final response generation
require no further model calls.

The two results are saved separately in the
active Colab runtime.


In [6]:

case_1_message = "My package hasn't shown up."

case_1_config = {
    "configurable": {
        "thread_id": "langgraph-delivery-case"
    }
}

if "case_1_result" not in globals():
    case_1_result = graph.invoke(
        {
            "message": case_1_message,
            "log": []
        },
        config=case_1_config
    )

    print("Gemini workflow completed.")
else:
    print("Using saved workflow result.")

print("\nCustomer:", case_1_message)
print("Gemini category:", case_1_result["category"])
print("Selected policy:", case_1_result["policy"])
print("Final response:", case_1_result["answer"])

print("\nExecution log:")
for entry in case_1_result["log"]:
    print("-", entry)

assert case_1_result["category"] in ALLOWED_CATEGORIES
assert case_1_result["answer"].strip()

print("\nWorkflow execution: PASSED")


Gemini workflow completed.

Customer: My package hasn't shown up.
Gemini category: delivery
Selected policy: Standard delivery usually takes 3 to 5 business days. Contact support for help with delayed parcels.
Final response: According to the relevant support policy: Standard delivery usually takes 3 to 5 business days. Contact support for help with delayed parcels.

Execution log:
- Gemini classified request: delivery
- Executed delivery policy node
- Final response constructed

Workflow execution: PASSED


In [7]:

case_2_message = "My headphones arrived in pieces."

case_2_config = {
    "configurable": {
        "thread_id": "langgraph-damage-case"
    }
}

if "case_2_result" not in globals():
    case_2_result = graph.invoke(
        {
            "message": case_2_message,
            "log": []
        },
        config=case_2_config
    )

    print("Gemini workflow completed.")
else:
    print("Using saved workflow result.")

print("\nCustomer:", case_2_message)
print("Gemini category:", case_2_result["category"])
print("Selected policy:", case_2_result["policy"])
print("Final response:", case_2_result["answer"])

print("\nExecution log:")
for entry in case_2_result["log"]:
    print("-", entry)

assert case_2_result["category"] in ALLOWED_CATEGORIES
assert case_2_result["answer"].strip()

print("\nWorkflow execution: PASSED")


Gemini workflow completed.

Customer: My headphones arrived in pieces.
Gemini category: damage
Selected policy: Customers who receive damaged items should contact support to discuss replacement options.
Final response: According to the relevant support policy: Customers who receive damaged items should contact support to discuss replacement options.

Execution log:
- Gemini classified request: damage
- Executed damage policy node
- Final response constructed

Workflow execution: PASSED



## 6. Evaluate Decisions and Inspect Graph State

We will now evaluate both workflows locally.

We want to know:

1. Did Gemini select the expected category?
2. Did LangGraph execute the matching policy branch?
3. Did the selected policy match the final response?
4. Were the two execution histories kept separate?

We use the expected labels defined before
executing the model.

This evaluation does not call Gemini again.

We also inspect the saved execution state to
verify that the graph retained the classification
and resulting policy selection.

Checkpoint inspection is different from model
streaming: it examines previously saved execution
information without running the model again.


In [8]:

TEST_RESULTS = [
    {
        "name": "Delivery case",
        "result": case_1_result,
        "expected": "delivery",
        "config": case_1_config
    },
    {
        "name": "Damage case",
        "result": case_2_result,
        "expected": "damage",
        "config": case_2_config
    }
]

classification_passes = 0

for test in TEST_RESULTS:
    result = test["result"]
    expected = test["expected"]

    classification_ok = (
        result["category"] == expected
    )

    policy_ok = (
        result["policy"] == POLICIES[result["category"]]
    )

    branch_ok = (
        f"Executed {result['category']} policy node"
        in result["log"]
    )

    snapshot = graph.get_state(test["config"])

    checkpoint_ok = (
        snapshot.values["category"] == result["category"]
        and snapshot.next == ()
    )

    classification_passes += int(classification_ok)

    print("\n" + test["name"])
    print("Expected:", expected)
    print("Actual:", result["category"])
    print("Classification correct:", classification_ok)
    print("Policy consistent:", policy_ok)
    print("Correct branch executed:", branch_ok)
    print("Checkpoint valid:", checkpoint_ok)

    assert policy_ok
    assert branch_ok
    assert checkpoint_ok


print(
    "\nClassification accuracy:",
    f"{classification_passes}/{len(TEST_RESULTS)}"
)

history_1 = list(graph.get_state_history(case_1_config))
history_2 = list(graph.get_state_history(case_2_config))

print("\nDelivery checkpoints:", len(history_1))
print("Damage checkpoints:", len(history_2))

assert len(history_1) > 1
assert len(history_2) > 1

print("\nLocal graph verification completed.")
print("Additional Gemini requests: 0")



Delivery case
Expected: delivery
Actual: delivery
Classification correct: True
Policy consistent: True
Correct branch executed: True
Checkpoint valid: True

Damage case
Expected: damage
Actual: damage
Classification correct: True
Policy consistent: True
Correct branch executed: True
Checkpoint valid: True

Classification accuracy: 2/2

Delivery checkpoints: 5
Damage checkpoints: 5

Local graph verification completed.
Additional Gemini requests: 0



## Findings and Limitations

### What This Notebook Demonstrates

- Designing an LLM classification prompt.
- Defining allowed output categories and rules.
- Using `ChatPromptTemplate` with Gemini.
- Building a graph with `StateGraph`.
- Sharing data across nodes using `TypedDict`.
- Routing execution with conditional edges.
- Using real model predictions to select graph branches.
- Selecting policy information without additional
  model requests.
- Inspecting saved graph execution state.
- Evaluating model decisions against reference labels.

### Prompt Engineering Techniques

**Role definition**

Gemini is instructed to act as a customer-support
request classifier.

**Category definitions**

The prompt explains the meaning of each permitted
classification label.

**Constrained output**

The model is instructed to return exactly one
lowercase category.

**Output validation**

Python rejects categories not included in the
application's allowed values.

**Evaluation**

The model's selected category is compared with
a predefined expected answer.

### LangGraph's Responsibility

Gemini decides which category best describes
the customer message.

LangGraph uses that validated decision to select
the appropriate policy-processing branch.

This separates model inference from deterministic
workflow control.

### Relationship to Notebook 13

Notebook 13 uses Gemini to generate answers
from retrieved policy evidence.

Notebook 14 uses Gemini to determine which
workflow branch executes.

The two notebooks therefore demonstrate
different roles for language models.

### Limitations

- Only two synthetic customer messages are evaluated.
- The experiment cannot establish general
  classification accuracy.
- The model returns a text label rather than a
  provider-enforced structured-output object.
- The application validates the label locally.
- Policies are fictional and read-only.
- The final response is constructed by Python,
  not generated by another language-model call.
- The workflow does not perform real customer actions.
- In-memory checkpoints are lost when the runtime ends.
- This notebook does not implement human approval,
  tool execution or durable persistence.
- The two test messages do not cover every
  category or ambiguous multi-issue requests.

### Observed Results

Two customer-support requests were classified using
Gemini through a LangChain prompt pipeline.

**Case 1 — Delivery**

- Customer message: "My package hasn't shown up."
- Expected category: `delivery`
- Gemini category: `delivery`
- LangGraph executed the delivery policy branch.

**Case 2 — Damage**

- Customer message: "My headphones arrived in pieces."
- Expected category: `damage`
- Gemini category: `damage`
- LangGraph executed the damage policy branch.

**Evaluation summary**

- Classification accuracy: 2/2.
- Correct policy branches executed: 2/2.
- Both final responses matched their selected policies.
- Five checkpoints were recorded for each execution thread.
- All local graph verification assertions passed.

These results demonstrate correct model-driven routing
on the two synthetic examples. They do not establish
general classification accuracy.

### Conclusion

This notebook demonstrates prompt-engineered
decision-making inside a stateful LangGraph workflow.

A language model classifies customer requests
according to explicit prompt instructions.

LangGraph then uses the model's validated decision
to route processing through different nodes.

The experiment separates three responsibilities:

- Prompt engineering defines the decision criteria.
- Gemini makes the classification decision.
- LangGraph executes the selected workflow path.

Results should be interpreted according to the
two evaluated examples rather than generalized
to all customer-support requests.
